# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process a dataset defined with the Croissant standard using the `mlcroissant` library.

### Dataset Source
The dataset source is described by a Croissant schema at the following URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All references to record sets, fields, and columns are made via their Croissant `@id`.

In [ ]:
# List the available record sets (@id) with their fields and field @id
print("Available record sets (by @id):")
for recset in metadata.record_sets:
    print(f"- RecordSet @id: {recset.id} | name: {recset.name} | description: {recset.description}")
    print("  Fields:")
    for field in recset.fields:
        print(f"    - Field @id: {field.id} | name: {field.name} | dataType: {field.data_type}")
    print("  Columns (if present):")
    for col in getattr(recset, 'columns', []):
        print(f"    - Column @id: {col.id} | name: {col.name} (field: {col.field.id if col.field else 'n/a'})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Always reference record sets, fields, and columns by their `@id`.

In [ ]:
# Identify all record sets to extract data from (by @id)
record_set_ids = [rs.id for rs in metadata.record_sets]
print(f"All record set @ids: {record_set_ids}")

# For demonstration, we'll use the primary data record set: This is typically the main survey table.
# (Change this ID if a different table is desired.)
main_record_set_id = record_set_ids[0]

dataframes = {}
for record_set_id in record_set_ids:
    # Load all records for this record set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set {record_set_id}")

# Display columns (field @ids) of the main record set
print(f"Fields (@id) in record set {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields,
and grouping by a demographic or survey field. This prepares the data for further statistical or machine learning analysis.


In [ ]:
# --- Example: Filtering, normalization, and grouping using fields by @id ---
import numpy as np

# Pick a main numeric field for filtering and normalization (example: GAD-7 total score, field @id)
# Find the @id of the GAD-7 total score field from the schema listing above

# Update these example IDs to match your actual schema as desired:
numeric_field_id = None
group_field_id = None

# Here is an example to autopick a likely numeric field:
main_df = dataframes[main_record_set_id]
found_numeric = False
for f in metadata.record_sets[0].fields:
    if f.data_type in ["Integer", "Float", "Number"] and "gad" in f.id.lower():
        numeric_field_id = f.id
        found_numeric = True
        break
# If none found, just pick the first numeric field
if not found_numeric:
    for f in metadata.record_sets[0].fields:
        if f.data_type in ["Integer", "Float", "Number"]:
            numeric_field_id = f.id
            break
# Pick a reasonable group field, e.g., gender or village
for f in metadata.record_sets[0].fields:
    if f.data_type == "Text" and ("gender" in f.id.lower() or "village" in f.id.lower()):
        group_field_id = f.id
        break

print(f"Using numeric field: {numeric_field_id}")
print(f"Using group field: {group_field_id}")

# Continue only if the field exists and is present in the DataFrame
if numeric_field_id in main_df.columns:
    # Coerce the column to numeric if necessary (Croissant may load as object)
    main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors="coerce")
    # Filter for response values above an example threshold (e.g., moderate-severe symptoms)
    threshold = 10
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())
    # Normalize this numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Group by group_field_id (e.g., gender or village)
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No suitable numeric field found for analysis. Please update the field id based on your schema.")

## 5. Visualization
Visualize distributions of the selected numeric field and grouped means across a demographic category.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in main_df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id and group_field_id in main_df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=main_df[group_field_id], y=main_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No visualization possible, field not found.")

## 6. Conclusion
This notebook demonstrated loading a FAIR-compliant Croissant dataset, exploring the available record sets and fields, extracting main data tables, filtering and transforming the data, and visualizing outcomes using record set, field, and column `@id`s.

**Key findings and EDA techniques can be summarized here upon analyzing your dataset.**